In [1]:
!pip install shap lime xgboost -q

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/ashutoshdwivedi/.cache/kagglehub/datasets/clmentbisaillon/fake-and-real-news-dataset/versions/1


In [5]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context


import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

sns.set_style("whitegrid")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ashutoshdwivedi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ashutoshdwivedi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ashutoshdwivedi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [6]:
import pandas as pd

fake_df = pd.read_csv(f"{path}/Fake.csv")
true_df = pd.read_csv(f"{path}/True.csv")

fake_df['label'] = 0  # fake
true_df['label'] = 1  # real

df = pd.concat([fake_df, true_df], axis=0).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(df.shape)
df.head()

(44898, 5)


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [ ]:
import os
print(os.listdir(path))

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
print(df['label'].value_counts())

import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='label', data=df)
plt.title('Class Distribution (0 = Fake, 1 = Real)')
plt.show()

In [ ]:
print(df.isnull().sum())

In [ ]:
df['text_length'] = df['text'].astype(str).apply(len)

plt.figure(figsize=(8,5))
sns.histplot(data=df, x='text_length', hue='label', bins=50, kde=True)
plt.title('Article Length Distribution by Label')
plt.xlim(0, 10000)
plt.show()

df.groupby('label')['text_length'].describe()

In [ ]:
if 'subject' in df.columns:
    plt.figure(figsize=(10,5))
    sns.countplot(y='subject', hue='label', data=df)
    plt.title('Subject Distribution by Label')
    plt.show()


In [ ]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

In [ ]:
df['content'] = df['title'].astype(str) + ' ' + df['text'].astype(str)
df['clean_content'] = df['content'].apply(clean_text)

df[['content', 'clean_content']].head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_content'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape, X_test_tfidf.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
lr_preds = lr.predict(X_test_tfidf)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds))

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_preds = nb.predict(X_test_tfidf)

print("Naive Bayes")
print("Accuracy:", accuracy_score(y_test, nb_preds))
print(classification_report(y_test, nb_preds))

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, eval_metric='logloss', random_state=42)
xgb.fit(X_train_tfidf, y_train)
xgb_preds = xgb.predict(X_test_tfidf)

print("XGBoost")
print("Accuracy:", accuracy_score(y_test, xgb_preds))
print(classification_report(y_test, xgb_preds))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(18,5))
for ax, preds, name in zip(axes, [lr_preds, nb_preds, xgb_preds], ['Logistic Regression', 'Naive Bayes', 'XGBoost']):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Fake','Real']).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
import shap

# TF-IDF is sparse — convert a small sample to dense for SHAP
sample_size = 150
X_test_sample = X_test_tfidf[:sample_size]

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_sample)

print(shap_values.shape)


In [ ]:
feature_names = tfidf.get_feature_names_out()

X_test_sample_dense = X_test_sample.toarray()

shap.summary_plot(shap_values, X_test_sample_dense, feature_names=feature_names, max_display=20)

In [ ]:
idx = 0  # first sample in the test subset — change this to inspect different articles

print("Actual label:", y_test.iloc[idx], "| Predicted:", xgb_preds[idx])

shap.force_plot(
    explainer.expected_value,
    shap_values[idx],
    X_test_sample_dense[idx],
    feature_names=feature_names,
    matplotlib=True
)

In [ ]:
import joblib

joblib.dump(xgb, 'fake_news_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')



In [ ]:
xgb.save_model('fake_news_model.json')

